# 📓 Notebook Diff Explainer

**Zwei Jupyter-Notebooks vergleichen — visuell, strukturiert und optional mit LLM-Analyse.**

Dieses Notebook stellt eine **Gradio-Webanwendung** bereit, mit der zwei `.ipynb`-Dateien
interaktiv hochgeladen und verglichen werden können. Die Unterschiede werden in einer
farbcodierten Side-by-Side-Darstellung angezeigt (inspiriert von
[ipynbcompare.com](https://www.ipynbcompare.com/)).

### Aufbau

| Abschnitt | Inhalt |
|---|---|
| 1 – 3 | Grundlagen: Imports, Datenmodell, Notebook einlesen |
| 4 – 6 | Kern-Logik: Ähnlichkeit, Alignment, Änderungserkennung |
| 7 – 8 | Hilfsfunktionen und HTML-Visualisierung |
| 9 – 10 | Optionale Erweiterungen: nbdime, LLM-Analyse |
| 11 | Gradio-Interface starten |

> **Anleitung:** Alle Zellen der Reihe nach ausführen.  
> Die letzte Zelle startet die Gradio-App im Browser.

## 1 · Voraussetzungen & Installation

Die folgenden Pakete werden benötigt. `nbdime` ist **optional** — wird es nicht
gefunden, wird der entsprechende Schritt übersprungen.

In [ ]:
# Nur ausführen, falls Pakete noch nicht installiert sind:
#%pip install gradio nbformat requests
#%pip install nbdime   # optional

## 2 · Imports

Standardbibliotheken oben, externe Pakete darunter.

In [ ]:
from __future__ import annotations

import json
import html as html_module          # HTML-Sonderzeichen escapen
from dataclasses import dataclass, field
from difflib import SequenceMatcher, unified_diff
from pathlib import Path
from typing import Any, Optional

import gradio as gr
import nbformat
import requests

print("✔ Alle Imports erfolgreich.")

## 3 · Datenmodell

Zwei Datenklassen bilden das Rückgrat der Vergleichslogik:

| Klasse | Zweck |
|---|---|
| `CellSummary` | Fasst eine einzelne Notebook-Zelle zusammen (Index, Typ, Quelltext, Outputs). |
| `Change` | Beschreibt eine erkannte Änderung zwischen zwei Zellen inkl. Status und Ähnlichkeit. |

In [ ]:
@dataclass
class CellSummary:
    """Zusammenfassung einer einzelnen Notebook-Zelle."""
    index: int
    cell_type: str              # "code" | "markdown" | "raw"
    source: str                 # Quelltext der Zelle
    outputs: list[str] = field(default_factory=list)


@dataclass
class Change:
    """Beschreibt eine erkannte Änderung zwischen zwei Zellen."""
    status: str                 # "unchanged" | "added" | "removed" | "modified_source" | …
    old_index: Optional[int]
    new_index: Optional[int]
    cell_type: str
    similarity: float
    old_source: str
    new_source: str
    old_outputs: list[str] = field(default_factory=list)
    new_outputs: list[str] = field(default_factory=list)

## 4 · Notebook einlesen

`read_notebook` liest eine `.ipynb`-Datei und extrahiert für jede Zelle
Typ, Quelltext und Outputs.

> **Hinweis:** Jupyter kennt verschiedene Output-Typen (`stream`, `execute_result`,
> `display_data`, `error`). Die Funktion prüft `output_type` explizit,
> damit kein Fall doppelt oder gar nicht erfasst wird.

In [ ]:
def read_notebook(path: str | Path) -> list[CellSummary]:
    """Liest ein Notebook und gibt eine Liste von CellSummary-Objekten zurück."""
    nb = nbformat.read(str(path), as_version=4)
    cells: list[CellSummary] = []

    for i, cell in enumerate(nb.cells):
        outputs: list[str] = []

        if cell.cell_type == "code":
            for out in cell.get("outputs", []):
                otype = out.get("output_type", "")

                if otype == "stream":
                    outputs.append(str(out.get("text", "")))

                elif otype in ("execute_result", "display_data"):
                    text = out.get("data", {}).get("text/plain", "")
                    if text:
                        outputs.append(str(text))

                elif otype == "error":
                    tb = out.get("traceback", [])
                    outputs.append("\n".join(str(line) for line in tb))

        source = str(cell.get("source", ""))
        cells.append(CellSummary(i, cell.cell_type, source, outputs))

    return cells

## 5 · Ähnlichkeitsberechnung & Zell-Alignment

Um festzustellen, welche Zellen einander entsprechen, wird ein **Greedy-Alignment**
auf Basis der `SequenceMatcher`-Ähnlichkeit durchgeführt:

1. Für jede alte Zelle wird die ähnlichste noch nicht zugeordnete neue Zelle gesucht.
2. Liegt die Ähnlichkeit über `threshold`, wird das Paar gebildet.
3. Übrige neue Zellen gelten als *hinzugefügt*.

> ⚠️ **Einschränkung:** Der Greedy-Ansatz kann bei stark umgestellten Notebooks
> suboptimale Zuordnungen liefern. Für exakte Ergebnisse wäre ein
> [Hungarian-Algorithmus](https://en.wikipedia.org/wiki/Hungarian_algorithm) nötig.

In [ ]:
def similarity(a: CellSummary, b: CellSummary, ignore_outputs: bool = True) -> float:
    """Berechnet die Ähnlichkeit zweier Zellen (0.0 … 1.0).

    Zellen unterschiedlichen Typs erhalten automatisch 0.0.
    """
    if a.cell_type != b.cell_type:
        return 0.0

    text_a = a.source
    text_b = b.source

    if not ignore_outputs:
        text_a += "\n" + "\n".join(a.outputs)
        text_b += "\n" + "\n".join(b.outputs)

    return SequenceMatcher(None, text_a, text_b).ratio()


def best_alignment(
    old_cells: list[CellSummary],
    new_cells: list[CellSummary],
    ignore_outputs: bool = True,
    threshold: float = 0.55,
) -> list[tuple[Optional[CellSummary], Optional[CellSummary], float]]:
    """Ordnet alte und neue Zellen einander zu (Greedy-Verfahren).

    Parameters
    ----------
    threshold : float
        Mindest-Ähnlichkeit, damit zwei Zellen als zusammengehörig gelten.
    """
    used_new: set[int] = set()
    pairs: list[tuple[Optional[CellSummary], Optional[CellSummary], float]] = []

    for old_cell in old_cells:
        best_j: Optional[int] = None
        best_score = 0.0

        for j, new_cell in enumerate(new_cells):
            if j in used_new:
                continue
            score = similarity(old_cell, new_cell, ignore_outputs)
            if score > best_score:
                best_j = j
                best_score = score

        if best_j is not None and best_score >= threshold:
            used_new.add(best_j)
            pairs.append((old_cell, new_cells[best_j], best_score))
        else:
            pairs.append((old_cell, None, 0.0))

    # Neue Zellen ohne Partner → hinzugefügt
    for j, new_cell in enumerate(new_cells):
        if j not in used_new:
            pairs.append((None, new_cell, 0.0))

    return pairs

## 6 · Änderungen erkennen

Aus jedem Alignment-Paar wird ein `Change`-Objekt mit passendem Status erzeugt:

| Situation | Status |
|---|---|
| Beide Zellen vorhanden, identisch | `unchanged` |
| Quelltext geändert | `modified_source` |
| Nur Outputs geändert | `modified_outputs` |
| Beides geändert | `modified_code_and_outputs` |
| Nur alte Zelle vorhanden | `removed` |
| Nur neue Zelle vorhanden | `added` |

In [ ]:
def classify_change(
    old_cell: Optional[CellSummary],
    new_cell: Optional[CellSummary],
    score: float,
) -> Change:
    """Erzeugt ein Change-Objekt aus einem Alignment-Paar."""

    if old_cell and new_cell:
        source_equal  = old_cell.source  == new_cell.source
        outputs_equal = old_cell.outputs == new_cell.outputs

        if source_equal and outputs_equal:
            status = "unchanged"
        elif not source_equal and not outputs_equal:
            status = "modified_code_and_outputs"
        elif not source_equal:
            status = "modified_source"
        else:
            status = "modified_outputs"

        return Change(
            status=status,
            old_index=old_cell.index, new_index=new_cell.index,
            cell_type=old_cell.cell_type, similarity=score,
            old_source=old_cell.source, new_source=new_cell.source,
            old_outputs=old_cell.outputs, new_outputs=new_cell.outputs,
        )

    if old_cell:
        return Change(
            status="removed",
            old_index=old_cell.index, new_index=None,
            cell_type=old_cell.cell_type, similarity=0.0,
            old_source=old_cell.source, new_source="",
            old_outputs=old_cell.outputs, new_outputs=[],
        )

    assert new_cell is not None, "Mindestens eine Zelle muss vorhanden sein."
    return Change(
        status="added",
        old_index=None, new_index=new_cell.index,
        cell_type=new_cell.cell_type, similarity=0.0,
        old_source="", new_source=new_cell.source,
        old_outputs=[], new_outputs=new_cell.outputs,
    )

## 7 · Hilfsfunktionen

- `cell_diff_text` — klassischer Unified-Diff zwischen zwei Quelltexten.
- `short_preview` — kürzt langen Text auf eine einzeilige Vorschau.
- `compare_notebooks` — Orchestrierungsfunktion, die alles zusammenführt.

In [ ]:
def cell_diff_text(old_source: str, new_source: str) -> str:
    """Erzeugt einen Unified-Diff zwischen altem und neuem Quelltext."""
    old_lines = old_source.splitlines(keepends=True)
    new_lines = new_source.splitlines(keepends=True)
    diff = unified_diff(old_lines, new_lines, fromfile="alt", tofile="neu")
    return "".join(diff).strip() or "(kein Unterschied im Quelltext)"


def short_preview(text: str, max_len: int = 120) -> str:
    """Einzeilige Kurzvorschau eines Textes."""
    text = " ".join(text.strip().split())
    if not text:
        return "(leer)"
    return text[:max_len] + ("…" if len(text) > max_len else "")


def compare_notebooks(
    old_path: str | Path,
    new_path: str | Path,
    ignore_outputs: bool = True,
    threshold: float = 0.55,
) -> list[Change]:
    """Vergleicht zwei Notebooks und gibt eine Liste von Änderungen zurück."""
    old_cells = read_notebook(old_path)
    new_cells = read_notebook(new_path)
    aligned   = best_alignment(old_cells, new_cells, ignore_outputs, threshold)
    return [classify_change(oc, nc, score) for oc, nc, score in aligned]

## 8 · HTML-Visualisierung (Side-by-Side)

Die Kernfunktion `render_comparison_html` erzeugt eine **farbcodierte
Side-by-Side-Darstellung** des Vergleichs, die direkt in Gradio angezeigt wird:

| Farbe | Bedeutung |
|---|---|
| 🟢 Grün | Zelle hinzugefügt |
| 🔴 Rot | Zelle entfernt |
| 🟡 Gelb | Zelle geändert (Quelltext und/oder Outputs) |
| ⚪ Grau | Zelle unverändert (eingeklappt) |

Innerhalb geänderter Zellen werden die Unterschiede mit **Inline-Diff-Highlighting**
auf Zeilenebene hervorgehoben.

In [ ]:
def _esc(text: str) -> str:
    """HTML-Sonderzeichen escapen."""
    return html_module.escape(text)


def _inline_diff_html(old_src: str, new_src: str, context: int = 3) -> tuple[str, str]:
    """Kontextbasiertes Inline-Diff-Highlighting (wie GitHub).

    Statt den gesamten Quelltext zu zeigen, werden nur die geänderten
    Regionen mit 'context' Zeilen Umgebung dargestellt. Unveränderte
    Bereiche dazwischen werden als '⋯ N Zeilen unverändert ⋯' eingeklappt.
    """
    old_lines = old_src.splitlines()
    new_lines = new_src.splitlines()
    matcher   = SequenceMatcher(None, old_lines, new_lines)
    opcodes   = matcher.get_opcodes()

    # Schritt 1: Bestimme, welche Zeilen "interessant" sind (geändert + Kontext)
    old_show: set[int] = set()
    new_show: set[int] = set()

    for tag, i1, i2, j1, j2 in opcodes:
        if tag != "equal":
            # Die geänderten Zeilen selbst
            old_show.update(range(i1, i2))
            new_show.update(range(j1, j2))
            # Plus Kontext drumherum
            old_show.update(range(max(0, i1 - context), min(len(old_lines), i2 + context)))
            new_show.update(range(max(0, j1 - context), min(len(new_lines), j2 + context)))

    # Schritt 2: HTML erzeugen mit eingeklappten Bereichen
    left_parts:  list[str] = []
    right_parts: list[str] = []
    SKIP = '<span class="diff-skip">⋯ {n} Zeilen unverändert ⋯</span>'

    for tag, i1, i2, j1, j2 in opcodes:
        if tag == "equal":
            # Nur die Zeilen zeigen, die im Kontext-Fenster liegen
            skipped_left  = 0
            skipped_right = 0
            for offset, k in enumerate(range(i1, i2)):
                k_new = j1 + offset
                if k in old_show:
                    if skipped_left > 0:
                        left_parts.append(SKIP.format(n=skipped_left))
                        right_parts.append(SKIP.format(n=skipped_right))
                        skipped_left = 0
                        skipped_right = 0
                    esc = _esc(old_lines[k])
                    left_parts.append(esc)
                    right_parts.append(esc)
                else:
                    skipped_left  += 1
                    skipped_right += 1
            if skipped_left > 0:
                left_parts.append(SKIP.format(n=skipped_left))
                right_parts.append(SKIP.format(n=skipped_right))

        elif tag == "replace":
            for line in old_lines[i1:i2]:
                left_parts.append(f'<span class="diff-del">{_esc(line)}</span>')
            for line in new_lines[j1:j2]:
                right_parts.append(f'<span class="diff-ins">{_esc(line)}</span>')

        elif tag == "delete":
            for line in old_lines[i1:i2]:
                left_parts.append(f'<span class="diff-del">{_esc(line)}</span>')

        elif tag == "insert":
            for line in new_lines[j1:j2]:
                right_parts.append(f'<span class="diff-ins">{_esc(line)}</span>')

    return ("\n".join(left_parts), "\n".join(right_parts))


def _status_label(status: str) -> tuple[str, str]:
    """Gibt (Label-Text, CSS-Klasse) für einen Change-Status zurück."""
    mapping = {
        "added":                     ("Hinzugefügt",             "badge-added"),
        "removed":                   ("Entfernt",                "badge-removed"),
        "modified_source":           ("Quelltext geändert",      "badge-modified"),
        "modified_outputs":          ("Outputs geändert",        "badge-modified"),
        "modified_code_and_outputs": ("Code & Outputs geändert", "badge-modified"),
        "unchanged":                 ("Unverändert",             "badge-unchanged"),
    }
    return mapping.get(status, (status, "badge-unchanged"))


def _cell_type_badge(cell_type: str) -> str:
    """Kleines HTML-Badge für den Zelltyp."""
    icons = {"code": "⌨", "markdown": "📝", "raw": "📄"}
    icon  = icons.get(cell_type, "")
    return f'<span class="cell-type-badge">{icon} {cell_type}</span>'

In [ ]:
# ── CSS-Stylesheet für die Vergleichsdarstellung (helles Theme) ──

DIFF_CSS = """
<style>
.nb-diff-container {
    font-family: -apple-system, "Segoe UI", Helvetica, Arial, sans-serif;
    max-width: 100%;
    background: #ffffff;
    border: 1px solid #d1d9e0;
    border-radius: 8px;
    overflow: hidden;
    color: #1f2328;
}

/* Header */
.nb-diff-header {
    background: #f6f8fa;
    padding: 16px 20px;
    border-bottom: 1px solid #d1d9e0;
}
.nb-diff-header h2 {
    margin: 0 0 10px; font-size: 17px; font-weight: 600; color: #1f2328;
}
.nb-stats-bar { display: flex; gap: 18px; flex-wrap: wrap; }
.nb-stat {
    display: flex; align-items: center; gap: 6px;
    font-size: 13px; color: #59636e;
}
.nb-stat .dot {
    width: 10px; height: 10px; border-radius: 50%; display: inline-block;
}
.dot-added    { background: #1a7f37; }
.dot-removed  { background: #cf222e; }
.dot-modified { background: #bf8700; }
.dot-unchanged{ background: #8c959f; }

/* Column headers */
.nb-col-headers {
    display: grid; grid-template-columns: 1fr 1fr;
    border-bottom: 1px solid #d1d9e0; background: #f6f8fa;
}
.nb-col-header {
    padding: 8px 16px; font-size: 12px; font-weight: 600;
    color: #59636e; text-transform: uppercase; letter-spacing: 0.5px;
}
.nb-col-header:first-child { border-right: 1px solid #d1d9e0; }

/* Change rows */
.nb-change-row { border-bottom: 1px solid #d1d9e0; }
.nb-change-row:last-child { border-bottom: none; }
.nb-change-bar {
    display: flex; align-items: center; gap: 8px;
    padding: 7px 16px; background: #f6f8fa; border-bottom: 1px solid #e8ecf0;
}

/* Badges */
.badge {
    font-size: 11px; font-weight: 600; padding: 2px 8px;
    border-radius: 12px; text-transform: uppercase; letter-spacing: 0.3px;
}
.badge-added    { background: #dafbe1; color: #116329; border: 1px solid #aceebb; }
.badge-removed  { background: #ffebe9; color: #82071e; border: 1px solid #ffcecb; }
.badge-modified { background: #fff8c5; color: #6f5600; border: 1px solid #eee09a; }
.badge-unchanged{ background: #f0f1f3; color: #8c959f; border: 1px solid #d1d9e0; }
.cell-type-badge { font-size: 11px; color: #59636e; }
.similarity-info { font-size: 11px; color: #8c959f; margin-left: auto; }

/* Side-by-side panels */
.nb-side-by-side { display: grid; grid-template-columns: 1fr 1fr; }
.nb-panel { padding: 10px 14px; overflow-x: auto; min-height: 28px; }
.nb-panel-left  { border-right: 1px solid #e8ecf0; background: #ffffff; }
.nb-panel-right { background: #ffffff; }

/* Status-specific backgrounds */
.nb-change-row.status-added .nb-panel-right   { background: #dafbe1; }
.nb-change-row.status-added .nb-panel-left    { background: #f6f8fa; }
.nb-change-row.status-removed .nb-panel-left  { background: #ffebe9; }
.nb-change-row.status-removed .nb-panel-right { background: #f6f8fa; }
.nb-change-row.status-modified_source .nb-panel-left,
.nb-change-row.status-modified_code_and_outputs .nb-panel-left,
.nb-change-row.status-modified_outputs .nb-panel-left  { background: #ffebe9; }
.nb-change-row.status-modified_source .nb-panel-right,
.nb-change-row.status-modified_code_and_outputs .nb-panel-right,
.nb-change-row.status-modified_outputs .nb-panel-right { background: #dafbe1; }

.nb-panel pre {
    margin: 0;
    font-family: "SFMono-Regular", Menlo, Monaco, Consolas, monospace;
    font-size: 12.5px; line-height: 1.6;
    white-space: pre-wrap; word-break: break-word;
    color: #1f2328;
}
.nb-panel .empty-note { color: #8c959f; font-style: italic; font-size: 12px; }

/* Inline diff highlights */
.diff-del {
    background: #ffc1ba; color: #82071e;
    text-decoration: line-through; text-decoration-color: #cf222e88;
    border-radius: 3px; padding: 1px 3px;
}
.diff-ins {
    background: #aceebb; color: #116329;
    border-radius: 3px; padding: 1px 3px;
}

/* Unchanged: collapsed */
.nb-unchanged-collapsed {
    text-align: center; padding: 5px; font-size: 12px;
    color: #8c959f; background: #f6f8fa;
}

/* Output section */
.nb-outputs-section { border-top: 1px dashed #d1d9e0; }
.nb-outputs-label {
    font-size: 11px; color: #59636e; padding: 4px 16px 0; font-weight: 600;
}

/* Collapsed unchanged lines in diff */
.diff-skip {
    display: block;
    text-align: center;
    color: #8c959f;
    font-style: italic;
    font-size: 11px;
    background: #f6f8fa;
    border-top: 1px dashed #d1d9e0;
    border-bottom: 1px dashed #d1d9e0;
    padding: 2px 0;
    margin: 2px 0;
}
</style>
"""


In [ ]:
def render_comparison_html(
    changes: list[Change],
    old_name: str = "Original",
    new_name: str = "Geändert",
    show_unchanged: bool = False,
) -> str:
    """Erzeugt vollständiges HTML für den Side-by-Side-Vergleich."""

    # ── Statistik ──
    stats = {
        "total":     len(changes),
        "unchanged": sum(1 for c in changes if c.status == "unchanged"),
        "added":     sum(1 for c in changes if c.status == "added"),
        "removed":   sum(1 for c in changes if c.status == "removed"),
        "modified":  sum(1 for c in changes if c.status.startswith("modified")),
    }

    parts: list[str] = [DIFF_CSS, '<div class="nb-diff-container">']

    # Header mit Statistik
    parts.append(f'''
    <div class="nb-diff-header">
        <h2>📓 Notebook-Vergleich</h2>
        <div class="nb-stats-bar">
            <span class="nb-stat"><span class="dot dot-unchanged"></span>{stats["unchanged"]} unverändert</span>
            <span class="nb-stat"><span class="dot dot-added"></span>{stats["added"]} hinzugefügt</span>
            <span class="nb-stat"><span class="dot dot-removed"></span>{stats["removed"]} entfernt</span>
            <span class="nb-stat"><span class="dot dot-modified"></span>{stats["modified"]} geändert</span>
            <span class="nb-stat">│ {stats["total"]} Zellen gesamt</span>
        </div>
    </div>
    ''')

    # Spaltenüberschriften
    parts.append(f'''
    <div class="nb-col-headers">
        <div class="nb-col-header">◀ {_esc(old_name)}</div>
        <div class="nb-col-header">{_esc(new_name)} ▶</div>
    </div>
    ''')

    def _truncate(text: str, limit: int = 60) -> str:
        """Kürzt Quelltext für added/removed-Zellen (nicht für Diffs)."""
        lines = text.splitlines()
        if len(lines) > limit:
            return "\n".join(lines[:limit]) + f"\n\n… ({len(lines) - limit} weitere Zeilen)"
        return text

    # Jede Änderung rendern
    for c in changes:
        label_text, badge_class = _status_label(c.status)
        type_badge = _cell_type_badge(c.cell_type)

        # Position
        pos_parts = []
        if c.old_index is not None:
            pos_parts.append(f"#{c.old_index}")
        if c.new_index is not None:
            pos_parts.append(f"#{c.new_index}")
        pos_str = " → ".join(pos_parts)

        # Ähnlichkeit
        sim_str = ""
        if c.similarity > 0 and c.status != "unchanged":
            sim_str = f'<span class="similarity-info">Ähnlichkeit: {c.similarity:.0%}</span>'

        # Unverändert: eingeklappt
        if c.status == "unchanged" and not show_unchanged:
            parts.append(f'''
            <div class="nb-change-row status-unchanged">
                <div class="nb-unchanged-collapsed">
                    ─── {type_badge}  Zelle {pos_str} · unverändert ───
                </div>
            </div>''')
            continue

        parts.append(f'<div class="nb-change-row status-{c.status}">')
        parts.append(f'''
            <div class="nb-change-bar">
                <span class="badge {badge_class}">{label_text}</span>
                {type_badge}
                <span style="color:#59636e;font-size:12px">Zelle {pos_str}</span>
                {sim_str}
            </div>''')

        # Side-by-side Panels
        parts.append('<div class="nb-side-by-side">')

        if c.status == "added":
            parts.append('<div class="nb-panel nb-panel-left"><span class="empty-note">—</span></div>')
            parts.append(f'<div class="nb-panel nb-panel-right"><pre>{_esc(_truncate(c.new_source))}</pre></div>')

        elif c.status == "removed":
            parts.append(f'<div class="nb-panel nb-panel-left"><pre>{_esc(_truncate(c.old_source))}</pre></div>')
            parts.append('<div class="nb-panel nb-panel-right"><span class="empty-note">—</span></div>')

        elif c.status.startswith("modified"):
            # Kontextbasierter Diff — zeigt ALLE Änderungen mit Kontext,
            # auch in langen Zellen (kein Abschneiden mehr!)
            left_html, right_html = _inline_diff_html(c.old_source, c.new_source)
            parts.append(f'<div class="nb-panel nb-panel-left"><pre>{left_html}</pre></div>')
            parts.append(f'<div class="nb-panel nb-panel-right"><pre>{right_html}</pre></div>')

        else:   # unchanged (bei show_unchanged=True)
            parts.append(f'<div class="nb-panel nb-panel-left"><pre>{_esc(_truncate(c.old_source))}</pre></div>')
            parts.append(f'<div class="nb-panel nb-panel-right"><pre>{_esc(_truncate(c.new_source))}</pre></div>')

        parts.append('</div>')  # nb-side-by-side

        # Output-Vergleich bei Bedarf
        if c.status in ("modified_outputs", "modified_code_and_outputs"):
            old_out = "\n---\n".join(o[:500] for o in c.old_outputs[:3]) or "(keine)"
            new_out = "\n---\n".join(o[:500] for o in c.new_outputs[:3]) or "(keine)"
            parts.append(f'''
            <div class="nb-outputs-section">
                <div class="nb-outputs-label">Outputs: {len(c.old_outputs)} → {len(c.new_outputs)} Blöcke</div>
                <div class="nb-side-by-side">
                    <div class="nb-panel nb-panel-left"><pre style="color:#59636e;font-size:11.5px">{_esc(old_out)}</pre></div>
                    <div class="nb-panel nb-panel-right"><pre style="color:#59636e;font-size:11.5px">{_esc(new_out)}</pre></div>
                </div>
            </div>''')

        parts.append('</div>')  # nb-change-row

    parts.append('</div>')  # nb-diff-container
    return "\n".join(parts)

## 9 · Optionale nbdime-Integration

Falls [`nbdime`](https://nbdime.readthedocs.io/) installiert ist, kann eine
zweite, unabhängige Diff-Perspektive erzeugt werden. Bei Fehlen von `nbdime`
wird der Schritt automatisch übersprungen.

In [ ]:
import sys
import re

def _ansi_to_html(text: str) -> str:
    """Konvertiert ANSI-Farbcodes in HTML-<span>-Tags.

    Die wichtigsten Codes von nbdime:
      [31m = rot   (gelöschte Zeilen)
      [32m = grün  (hinzugefügte Zeilen)
      [34m = blau  (Abschnitts-Überschriften)
      [1m  = fett
      [0m  = Reset
    """
    color_map = {
        "31": "#cf222e",   # rot
        "32": "#1a7f37",   # grün
        "33": "#9a6700",   # gelb
        "34": "#0550ae",   # blau
        "35": "#8250df",   # magenta
        "36": "#1b7c83",   # cyan
    }

    result: list[str] = []
    open_spans = 0

    # Jedes Token ist entweder eine ANSI-Sequenz oder normaler Text
    for part in re.split(r'(\x1b\[[0-9;]*m|\[\d+(?:;\d+)*m)', text):
        # Echte ANSI-Sequenz (\x1b[...)
        if part.startswith("\\x1b[") or part.startswith("\x1b["):
            codes = part.lstrip("\\x1b[").lstrip("\x1b[").rstrip("m").split(";")
            if "0" in codes or codes == [""]:
                result.append("</span>" * open_spans)
                open_spans = 0
            else:
                styles = []
                for code in codes:
                    if code == "1":
                        styles.append("font-weight:700")
                    elif code in color_map:
                        styles.append(f"color:{color_map[code]}")
                if styles:
                    result.append(f'<span style=\"{";".join(styles)}\">')
                    open_spans += 1
        # Roh-Format [34m etc. (nbdime pretty_print gibt diese aus)
        elif re.match(r'^\[\d+(?:;\d+)*m$', part):
            codes = part.strip("[]m").split(";")
            if "0" in codes:
                result.append("</span>" * open_spans)
                open_spans = 0
            else:
                styles = []
                for code in codes:
                    if code == "1":
                        styles.append("font-weight:700")
                    elif code in color_map:
                        styles.append(f"color:{color_map[code]}")
                if styles:
                    result.append(f'<span style=\"{";".join(styles)}\">')
                    open_spans += 1
        else:
            result.append(html_module.escape(part))

    result.append("</span>" * open_spans)
    return "".join(result)


def run_nbdiff_summary(old_path: str, new_path: str, ignore_outputs: bool = True) -> str:
    """Vergleicht zwei Notebooks über die nbdime-Python-API.

    Gibt formatiertes HTML zurück (ANSI-Farbcodes → HTML-Farben).
    """
    try:
        from nbdime.diffing.notebooks import diff_notebooks
    except ImportError:
        return (
            f"<pre>ℹ️ nbdime ist im aktuellen Python nicht importierbar.\n"
            f"   Aktiver Interpreter: {sys.executable}\n"
            f"   → Im Terminal ausführen:\n"
            f"     {sys.executable} -m pip install nbdime</pre>"
        )

    try:
        nb_old = nbformat.read(str(old_path), as_version=4)
        nb_new = nbformat.read(str(new_path), as_version=4)

        # Metadata vor dem Diff entfernen (wie --ignore-metadata)
        for nb in (nb_old, nb_new):
            nb.metadata = {}
            for cell in nb.cells:
                cell.metadata = {}

        if ignore_outputs:
            for nb in (nb_old, nb_new):
                for cell in nb.cells:
                    if cell.cell_type == "code":
                        cell.outputs = []
                        cell.execution_count = None

        diff = diff_notebooks(nb_old, nb_new)

        if not diff:
            return "<pre>✔ Keine Unterschiede erkannt (nbdime).</pre>"

        # Pretty-Print mit Farben
        raw_text = ""
        try:
            import io
            from nbdime.prettyprint import PrettyPrintConfig, pretty_print_notebook_diff

            stream = io.StringIO()
            cfg = PrettyPrintConfig(out=stream)
            pretty_print_notebook_diff("", "", nb_old, diff, cfg)
            raw_text = stream.getvalue().strip()
        except Exception:
            raw_text = json.dumps(diff, indent=2, ensure_ascii=False, default=str)

        if not raw_text:
            return "<pre>Keine nbdime-Ausgabe verfügbar.</pre>"

        # ANSI → HTML
        html_body = _ansi_to_html(raw_text[:12_000])

        return (
            '<div style="font-family:Menlo,Monaco,Consolas,monospace;font-size:12.5px;'
            'line-height:1.6;background:#f6f8fa;border:1px solid #d1d9e0;border-radius:8px;'
            'padding:16px;overflow-x:auto;color:#1f2328">'
            f'<pre style="margin:0;white-space:pre-wrap">{html_body}</pre></div>'
        )

    except Exception as exc:
        return f"<pre>⚠️ nbdime-Fehler: {html_module.escape(str(exc))}</pre>"

## 10 · Optionale LLM-Analyse

Die folgenden Funktionen senden die Änderungen an eine **OpenAI-kompatible API**
(z. B. LM Studio lokal oder OpenRouter) und lassen die Unterschiede semantisch erklären.

> **Dieser Schritt ist optional.** Der visuelle Vergleich funktioniert ohne LLM.

In [ ]:
def build_llm_payload(changes: list[Change], ignore_outputs: bool = True, limit: int = 8) -> dict[str, Any]:
    """Baut den JSON-Payload für die LLM-Anfrage."""
    interesting = [c for c in changes if c.status != "unchanged"][:limit]
    return {
        "task": "Explain semantic differences between two Jupyter notebooks.",
        "rules": [
            "Trenne Fakten von Hypothesen.",
            "Klassifiziere: refactoring | behavior_change | presentation | output_only.",
            "Weise auf Reproduzierbarkeits-Risiken hin.",
            "Schreibe knappes deutsches Markdown.",
        ],
        "ignore_outputs": ignore_outputs,
        "changes": [
            {
                "status": c.status, "cell_type": c.cell_type,
                "old_index": c.old_index, "new_index": c.new_index,
                "similarity": round(c.similarity, 3),
                "old_source": c.old_source[:2500], "new_source": c.new_source[:2500],
                "old_outputs": c.old_outputs[:2],  "new_outputs": c.new_outputs[:2],
                "unified_diff": cell_diff_text(c.old_source, c.new_source)[:2500],
            }
            for c in interesting
        ],
    }


def stream_llm(
    payload: dict[str, Any],
    api_base: str, api_key: str, model: str,
):
    """Streamt die LLM-Antwort von einer OpenAI-kompatiblen API."""
    url = api_base.rstrip("/") + "/chat/completions"
    prompt = (
        "Du bist ein erfahrener Python- und Notebook-Reviewer. "
        "Analysiere den strukturierten Notebook-Diff. "
        "Antworte in knappem, deutschem Markdown. Hypothesen klar kennzeichnen.\n\n"
        + json.dumps(payload, ensure_ascii=False, indent=2)
    )

    body = {
        "model": model,
        "messages": [
            {"role": "system", "content": "Sei präzise, skeptisch und knapp."},
            {"role": "user",   "content": prompt},
        ],
        "temperature": 0.2,
        "stream": True,
    }

    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    with requests.post(url, headers=headers, json=body, timeout=(10, 1800), stream=True) as r:
        r.raise_for_status()
        r.encoding = "utf-8"  # Server sendet oft kein charset → requests rät Latin-1
        for raw_line in r.iter_lines(decode_unicode=True):
            if not raw_line or not raw_line.startswith("data: "):
                continue
            data_str = raw_line[6:].strip()
            if data_str == "[DONE]":
                break
            try:
                event = json.loads(data_str)
            except json.JSONDecodeError:
                continue
            delta = event.get("choices", [{}])[0].get("delta", {})
            content = delta.get("content")
            if content:
                yield content

---

## 11 · Gradio-Interface starten

Die folgende Zelle baut das **Gradio-Interface** auf und startet es.
Nach dem Ausführen erscheint ein Link — klicke ihn an, um die App im Browser zu öffnen.

**Bedienung:**
1. Zwei `.ipynb`-Dateien hochladen (Original und Geändert).
2. Optionen wählen (Outputs ignorieren, LLM aktivieren, …).
3. Auf **Vergleichen** klicken.
4. Ergebnis im Tab *Notebook-Vergleich* ansehen.

In [ ]:
def compare_notebooks_gradio(
    old_file, new_file,
    ignore_outputs, show_unchanged, threshold,
    use_llm, api_mode, api_base, api_key, model,
):
    """Callback für den Gradio-Button. Gibt (HTML, Payload-JSON, LLM-Text) zurück."""

    # ── Validierung ──
    if old_file is None or new_file is None:
        raise gr.Error("Bitte beide Notebooks hochladen.")

    old_path = old_file.name if hasattr(old_file, "name") else str(old_file)
    new_path = new_file.name if hasattr(new_file, "name") else str(new_file)

    for label, p in [("Original", old_path), ("Geändert", new_path)]:
        if not Path(p).is_file():
            raise gr.Error(f"Datei nicht gefunden: {p}")

    # ── Vergleich ──
    changes = compare_notebooks(old_path, new_path, ignore_outputs, threshold)
    comparison_html = render_comparison_html(
        changes,
        old_name=Path(old_path).name,
        new_name=Path(new_path).name,
        show_unchanged=show_unchanged,
    )
    nbdime_text = run_nbdiff_summary(old_path, new_path, ignore_outputs)
    payload = build_llm_payload(changes, ignore_outputs)
    payload_json = json.dumps(payload, ensure_ascii=False, indent=2)

    # ── LLM (optional, mit Streaming) ──
    if not use_llm:
        yield comparison_html, nbdime_text, payload_json, "LLM-Erklärung deaktiviert."
        return

    if not model:
        raise gr.Error("Bitte ein Modell angeben, wenn LLM aktiviert ist.")

    if api_mode == "openrouter":
        effective_base = api_base or "https://openrouter.ai/api/v1"
    else:
        effective_base = api_base or "http://localhost:1234/v1"

    partial = "⏳ Starte LLM-Analyse …\n\n"
    yield comparison_html, nbdime_text, payload_json, partial

    try:
        for chunk in stream_llm(payload, effective_base, api_key or "", model):
            partial += chunk
            yield comparison_html, nbdime_text, payload_json, partial
    except Exception as exc:
        partial += f"\n\n❌ LLM-Aufruf fehlgeschlagen: {exc}"
        yield comparison_html, nbdime_text, payload_json, partial

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                     GRADIO-APP AUFBAUEN                        ║
# ╚══════════════════════════════════════════════════════════════════╝

with gr.Blocks(
    title="📓 Notebook Diff Explainer",
    theme=gr.themes.Base(
        primary_hue="blue",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Inter"),
    ),
    css="""
        .gradio-container { max-width: 1400px !important; }
        #compare-btn { min-height: 48px; font-size: 16px; }
    """,
) as demo:

    gr.Markdown(
        "# 📓 Notebook Diff Explainer\n"
        "Zwei Jupyter-Notebooks vergleichen und optional per LLM erklären lassen."
    )

    # ── Datei-Upload ──
    with gr.Row():
        old_file = gr.File(label="◀ Original (.ipynb)", file_types=[".ipynb"])
        new_file = gr.File(label="Geändert (.ipynb) ▶", file_types=[".ipynb"])

    # ── Vergleichs-Optionen ──
    with gr.Row():
        ignore_outputs = gr.Checkbox(value=True,  label="Outputs ignorieren")
        show_unchanged = gr.Checkbox(value=False, label="Unveränderte Zellen anzeigen")
        threshold      = gr.Slider(0.1, 1.0, value=0.55, step=0.05,
                                   label="Ähnlichkeits-Schwellenwert")

    # ── LLM-Optionen (eingeklappt) ──
    with gr.Accordion("🤖 LLM-Optionen (optional)", open=False):
        use_llm  = gr.Checkbox(value=False, label="LLM-Erklärung erzeugen")
        api_mode = gr.Dropdown(["local", "openrouter"], value="local", label="LLM-Ziel")
        with gr.Row():
            api_base = gr.Textbox(label="API Base URL",
                                  placeholder="http://localhost:1234/v1")
            api_key  = gr.Textbox(label="API Key", type="password",
                                  placeholder="bei lokalem Server leer lassen")
            model    = gr.Textbox(label="Modell",
                                  placeholder="z. B. qwen/qwen3-32b")

    run_btn = gr.Button("🔍 Vergleichen", variant="primary", elem_id="compare-btn")

    # ── Ergebnis-Tabs ──
    with gr.Tab("📊 Notebook-Vergleich"):
        result_html = gr.HTML()

    with gr.Tab("📋 nbdime-Ausgabe"):
        nbdime_box = gr.HTML(label="nbdime-Diff")

    with gr.Tab("📦 LLM-Payload"):
        payload_box = gr.Code(language="json", label="Payload")

    with gr.Tab("🤖 LLM-Erklärung"):
        llm_box = gr.Markdown()

    run_btn.click(
        compare_notebooks_gradio,
        inputs=[old_file, new_file, ignore_outputs, show_unchanged, threshold,
                use_llm, api_mode, api_base, api_key, model],
        outputs=[result_html, nbdime_box, payload_box, llm_box],
    )

# ── App starten ──
demo.queue().launch(inbrowser=True)